In [2]:
pip install xgboost

Note: you may need to restart the kernel to use updated packages.


In [3]:
#--- Import crucial libraries. --#
import os
import time
import numpy as np
import pandas as pd
import xgboost as xgb   
import matplotlib.pyplot as plt 
from IPython.display import display, clear_output
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

In [4]:
#-- Import the dataset. --#
df = pd.read_csv('../Processed/efficiency_stations.csv')

In [6]:
#-- Data Inspection and EDA. --#
print("Data Informatom.")
print(df.info())

Data Informatom.
<class 'pandas.DataFrame'>
RangeIndex: 3504 entries, 0 to 3503
Data columns (total 24 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   x                 3504 non-null   float64
 1   y                 3504 non-null   float64
 2   fid               3504 non-null   int64  
 3   objectid          3504 non-null   int64  
 4   tfm_id            3504 non-null   int64  
 5   tfm_desc          3504 non-null   str    
 6   tfm_typ_de        3504 non-null   str    
 7   movement_t        3500 non-null   str    
 8   site_desc         3504 non-null   str    
 9   road_nbr          3504 non-null   int64  
 10  declared_r        3504 non-null   str    
 11  local_road        3504 non-null   str    
 12  data_src_c        3504 non-null   str    
 13  data_sourc        3504 non-null   str    
 14  time_categ        3504 non-null   str    
 15  year_since        3504 non-null   int64  
 16  last_year         3504 non-null   in

### 📝 Feature Selection for Traffic Volume Prediction

For predict the traffic volume (**AADT**), our team select the following features based on their spatial and historical significance:

1. Spatial Coordinates (x, y): Traffic patterns are heavily dependent on geographic location.
2. Time Factor (last_year): Captures the historical growth and trends in traffic over time.
3. Efficiency Index: Provides a normalized context of station performance.
4. Target Variable (aadt_allve): The actual traffic volume we aim to predict.

In [7]:
#-- Dispaying data for model learning, training, and predicting. --#
training_cols = ['x', 'y', 'last_year', 'efficiency_index', 'aadt_allve']
display_df = df[training_cols].copy()

#-- Output the result with a clear description. --#
print("--- Prepared Data for Model Training (Feature & Target Subset) ---")
print(f"Total records ready for training: {len(display_df)}")
print("-" * 65)

#-- Display the first 10 rows to verify the values --#
display(display_df.head(10))

#-- Quick check on the data statistics for these specific columns. =-#
print("\n--- Summary Statistics for Training Data ---")
display(display_df.describe().round(2))


--- Prepared Data for Model Training (Feature & Target Subset) ---
Total records ready for training: 3504
-----------------------------------------------------------------


,x,y,last_year,efficiency_index,aadt_allve
0,144.560081,-37.968598,1998,1.00,40
1,145.088615,-37.881431,2006,2.43,13000
2,145.090058,-37.882231,1997,1.57,8400
3,145.088615,-37.881431,0,0.00,0
4,145.222981,-37.957762,2007,1.53,8600
5,145.223304,-37.957902,2004,1.46,8200
6,145.223576,-37.957983,2000,1.01,5700
7,145.173853,-37.918828,2006,2.00,9100
8,145.169904,-37.916506,0,0.00,0
9,145.039918,-37.843193,1989,2.33,43000



--- Summary Statistics for Training Data ---


,x,y,last_year,efficiency_index,aadt_allve
count,3504.00,3504.00,3504.00,3380.00,3504.00
mean,144.89,-37.62,1849.27,2.51,9241.48
std,1.11,0.65,538.75,1.88,13472.59
min,140.97,-38.94,0.00,0.00,0.00
25%,144.61,-37.94,1997.00,1.20,500.00
50%,144.97,-37.79,2008.00,2.03,4800.00
75%,145.19,-37.57,2013.00,3.32,12000.00
max,149.74,-34.18,2016.00,23.00,123000.00


In [8]:
pip install jinja2

Note: you may need to restart the kernel to use updated packages.


In [18]:
# --- 1. DATA PREPARATION (English Comments) ---#       
#-- Define columns for features, target, and display information. --#
display_info = ['road_name', 'tfm_id']
features = ['x', 'y', 'last_year', 'efficiency_index']
target = 'aadt_allve'
#-- Create a clean dataset to prevent 'Length Mismatch' and 'KeyError'. --#
#-- whichensures we have exactly the same rows for training and displaying --#
clean_df = df[display_info + features + [target]].dropna().copy()
X = clean_df[features]
y = clean_df[target]

In [19]:
# --- 2. MODEL TRAINING PHASE ---
print("--- Initializing Model Training... ---")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize XGBoost Regressor with optimized hyperparameters
xgb_model = xgb.XGBRegressor(
    n_estimators=100, 
    learning_rate=0.05, 
    max_depth=5, 
    random_state=42
)

# Fit the model on training data
xgb_model.fit(X_train, y_train)

# Calculate performance metrics for the dashboard header
val_preds = xgb_model.predict(X_test)
r2_val = r2_score(y_test, val_preds)
mae_val = mean_absolute_error(y_test, val_preds)
print("--- Training Complete. System Ready. ---")

--- Initializing Model Training... ---
--- Training Complete. System Ready. ---


In [20]:
# --- 3. TRAFFIC ENGINEERING LOGIC ---
FREE_FLOW_SPEED = 60      
JAM_VOLUME_THRESHOLD = 2200 
UPDATE_INTERVAL_SECONDS = 600 # 10-minute update cycle

def calculate_speed_logic(volume):
    """
    Implements Greenshields' Model for Variable Speed Limits (VSL).
    Dynamically adjusts speed to maximize network throughput.
    """
    if volume <= 0: return FREE_FLOW_SPEED
    reduction_factor = volume / (JAM_VOLUME_THRESHOLD * 1.5)
    optimal_speed = FREE_FLOW_SPEED * (1 - reduction_factor)
    return max(10, min(FREE_FLOW_SPEED, round(optimal_speed, 1)))

In [ ]:
# --- 4. LIVE SIMULATION LOOP ---
try:
    while True:
        # Perform real-time inference
        clean_df['predicted_volume'] = xgb_model.predict(X).round(0).astype(int)
        clean_df['optimal_speed_kmh'] = clean_df['predicted_volume'].apply(calculate_speed_logic)
        
        def define_status(v):
            if v > 1800: return "🔴 Congested"
            if v > 1000: return "🟡 Heavy"
            return "🟢 Normal"
        
        clean_df['traffic_status'] = clean_df['predicted_volume'].apply(define_status)

        # UI Update
        clear_output(wait=True)
        print(f"🕒 LIVE TRAFFIC MONITORING | SYSTEM TIME: {time.strftime('%H:%M:%S')}")
        print("=" * 105)
        print(f"📊 MODEL ACCURACY: R2 = {r2_val:.4f} | MAE = {mae_val:.2f}")
        print(f"📡 NETWORK NODES: {len(clean_df)} Sensors Active")
        print("=" * 105)

        # Prepare Styled Dashboard (Requires Jinja2 installed)
        dashboard_view = clean_df[display_info + ['predicted_volume', 'optimal_speed_kmh', 'traffic_status']]
        dashboard_view = dashboard_view.sort_values(by='predicted_volume', ascending=False)
        
        try:
            # Applying Heatmap coloring to the Predicted Volume column
            styled_table = dashboard_view.head(15).style.background_gradient(
                subset=['predicted_volume'], 
                cmap='YlOrRd'
            ).format({'optimal_speed_kmh': "{:.1f} km/h"})
            display(styled_table)
        except:
            # Fallback if Jinja2 is not yet recognized
            display(dashboard_view.head(15))

        # Network Announcements
        print("\n" + "="*35 + " NETWORK ANNOUNCEMENTS " + "="*35)
        congested = dashboard_view[dashboard_view['traffic_status'] == "🔴 Congested"]
        normal = dashboard_view[dashboard_view['traffic_status'] == "🟢 Normal"]
        
        if not congested.empty:
            print(f"⚠️  CRITICAL ALERTS: {len(congested)} road segments require speed intervention.")
            for _, row in congested.head(3).iterrows():
                print(f"   - {row['road_name']} (ID:{row['tfm_id']}) is CONGESTED. Advised Speed: {row['optimal_speed_kmh']} km/h")
        
        if not normal.empty:
            print(f"✅  NORMAL STATUS: {normal['road_name'].nunique()} roads are operating optimally.")

        # Professional 10-minute countdown timer
        print("\n" + "-"*105)
        for remaining in range(UPDATE_INTERVAL_SECONDS, 0, -1):
            mins, secs = divmod(remaining, 60)
            print(f"\r⏳ STANDBY: Next network-wide ingestion in {mins:02d}:{secs:02d} ", end="")
            time.sleep(1)

except KeyboardInterrupt:
    print("\n✅ System maintenance mode active. Simulation paused.")

SyntaxError: incomplete input (3618586648.py, line 59)